# Quantum Volume Characterization

Quantum volume is a benchmark for the effective performance of a quantum processor, capturing the impact of errors across multiqubit circuits. It is based on random circuits and the **heavy output probability**, namely the probability of measuring bitstrings whose ideal probabilities are above the median of the noiseless output distribution. For an ideal device this probability is approximately $0.85$. A width is considered successful if the measured heavy output probability exceeds $2/3$ with a $2\sigma$ confidence margin. The quantum volume is then defined as $\mathrm{QV}=2^n$, where $n$ is the largest width that passes.

***

### The model
For $n$ qubits, the model consists of $n$ layers. In each layer, the qubits are randomly paired, and a Haar-random SU(4) gate is applied to each pair. After all layers, the circuit is measured. The heavy outputs are determined from the noiseless distribution of the same random circuit, which is why the protocol is mainly practical for moderate-size QPUs.

### The protocol

For a fixed width $n$, we generate $T$ independent random model circuits of width $n$ and depth $n$, and execute them on the device. For each circuit, we estimate the heavy-output probability from the measurement data, namely the total probability assigned to outcomes whose ideal probabilities are above the median of the ideal output distribution. We then aggregate this quantity over the $T$ trials and construct a lower confidence bound.

When the number of completed trials is at least $100$, we use the pooled confidence criterion of Cross et al [[1]](#Cross_etal). For smaller runs, we use the simpler normal-approximation lower bound $\bar{h}-2\,\mathrm{SE}(\bar{h})$, where $\bar{h}$ is the mean heavy-output probability across trials and $\mathrm{SE}(\bar{h})$ is its standard error. A width $n$ is considered successful if the resulting lower bound exceeds $2/3$.

The reported quantum volume is $\mathrm{QV}=2^n$, where $n$ is the largest tested width that passes. In particular, if the largest executed width also passes, the true quantum volume is at least this value and may in fact be larger.

***
***

In [1]:
import sys

sys.path.insert(0, "..")

from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

### Choosing on which backend to run 

The quantum model is already defined in the `QuantumVolumeProtocol` class. We first define our hardware runners. Thus, we only need to
define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited after defining the `ResultCollector`.)*

In [2]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}


classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    # *ionq_runners,
    # *ibm_runners,
]

### Set a `QuantumVolumeProtocol` with a directory name fixed for the current run

We also need to pass the minimal and maximal number of qubits, and the number of trails (a suggested number is around 100, here we take only 10). 

In [3]:
from pathlib import Path

from protocol import QuantumVolumeProtocol

QV_DESCRIPTION = Path("../descriptions/quantum_volume.tex").read_text(encoding="utf-8")

protocol = QuantumVolumeProtocol(
    min_num_qubits=2,
    max_num_qubits=4,
    num_trials=10,
    runners=runners,
    results_dir="results/quantum_volume",
    max_submitted_jobs_in_dir=3,
    report_family_title="Quantum Volume",
    report_family_description=QV_DESCRIPTION,
    build_report_each_time=True,
)

We run the protocol. What happens in practice is that for a given number of qubits, all jobs are sent asynchronously. For convenience we can define a loop with some waiting time (see commented out code). 

In [4]:
import asyncio

from IPython.display import clear_output, display

await protocol.run();

# while True:
#     await protocol.run()

#     summary = protocol.all_width_summaries()

#     clear_output(wait=True)

#     if not summary.empty and (summary["num_completed"] == summary["num_trials_requested"]).all():
#         print("All trials completed.")
#         break

#     await asyncio.sleep(10)

Finally, we can print the summary of the results, as well as update the report.

In [5]:
display(protocol.all_width_summaries())
display(protocol.quantum_volume_summary())
display(protocol.final_summary())

,num_qubits,backend_service_provider,backend_name,num_trials_requested,num_completed,num_total,mean_score,std_score,stderr_score,lower_confidence_bound,passed
0,2,Classiq,simulator,10,10,10,0.783200,0.078078,0.024690,0.733819,True
1,2,Classiq,simulator_statevector,10,10,10,0.783517,0.076562,0.024211,0.735095,True
2,2,IBM Quantum,ibm_kingston,10,10,10,0.770000,0.091807,0.029032,0.711936,True
3,2,IonQ,qpu.forte-1,10,10,10,0.729200,0.098982,0.031301,0.666598,False
4,3,Classiq,simulator,10,10,10,0.795600,0.098567,0.031170,0.733261,True
5,3,Classiq,simulator_statevector,10,10,10,0.799250,0.099618,0.031502,0.736246,True
6,3,IBM Quantum,ibm_kingston,10,10,10,0.777500,0.095147,0.030088,0.717324,True
7,3,IonQ,qpu.forte-1,10,10,10,0.771700,0.090022,0.028468,0.714765,True
8,4,Classiq,simulator,10,10,10,0.821500,0.036924,0.011676,0.798147,True
9,4,Classiq,simulator_statevector,10,10,10,0.828553,0.040833,0.012912,0.802729,True


,backend_service_provider,backend_name,largest_passing_width,quantum_volume
0,Classiq,simulator,4,16
1,Classiq,simulator_statevector,4,16
2,IBM Quantum,ibm_kingston,4,16
3,IonQ,qpu.forte-1,4,16


,backend_service_provider,backend_name,largest_passing_width,quantum_volume,total_execution_time
0,Classiq,simulator,4,16,0.317528
1,Classiq,simulator_statevector,4,16,0.323045
2,IBM Quantum,ibm_kingston,4,16,12736.592649
3,IonQ,qpu.forte-1,4,16,371.183681


## Refrences

<a id='Cross_etal'>[1]</a>: [Andrew W. Cross, Lev S. Bishop, Sarah Sheldon, Paul D. Nation, Jay M. Gambetta. "Validating quantum computers using randomized model circuits." Physical Review A 100, 032328 (2019).](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.100.032328)